In [ ]:
from datasets import load_dataset

dataset = load_dataset("dinushiTJ/nz_research_commons_classification")

dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence'],
        num_rows: 13271
    })
    test: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence'],
        num_rows: 3318
    })
})

In [ ]:
def combine_text_for_embedding(example):
    # Define the fields to combine
    fields = ['title', 'authors', 'subjects', 'abstract', 'text', 'year']
    combined_parts = []

    for field in fields:
        if field in example and example[field] is not None:
            value = str(example[field]).strip()

            # Skip empty strings
            if value:
                combined_parts.append(f"{field}: {value}")

    example['embedding_text'] = "\n\n".join(combined_parts)
    return example

In [ ]:
# Apply the function to the dataset to create the new 'embedding_text' column
dataset_with_embedding_text = dataset.map(combine_text_for_embedding)

# Display a sample to verify the new column
print(dataset_with_embedding_text['train'][0]['embedding_text'])
# You can also inspect the dataset features to confirm the new column
print(dataset_with_embedding_text['train'].features)

title: The Te Aroha hot springs (mainly in the nineteenth century)

authors: Hart, Philip

abstract: Unique amongst New Zealand mining districts, the hot springs sited at the base of the mountain were popular with miners, residents, and an increasing number of visitors. Highly valued by Maori for their medicinal qualities, Pakeha visited them well before gold was discovered. Acquired by the Crown as part of the purchase of the Aroha Block, contention arose over whether the springs had been gifted to the Crown and whether Maori should be charged for using them. The provision of a small but free bath to Ngati Rahiri did not satisfy them.
 After Pakeha settled, the springs were developed and the surrounding domain was landscaped. Analysis of the water by experts produced claims about its curative qualities and many miracle cures were claimed, and the water was bottled until more recent analyses traced the existence of arsenic. Men reputedly skilled in hydropathy and similar ‘sciences’ wer

In [ ]:
print(dataset_with_embedding_text)

DatasetDict({
    train: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text'],
        num_rows: 13271
    })
    test: Dataset({
        features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text'],
        num_rows: 3318
    })
})


In [ ]:
# To view the content of the 'train' split, you can access it like this:
# You can change the index to view different rows
display(dataset_with_embedding_text['train'])

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence', 'embedding_text'],
    num_rows: 13271
})

In [ ]:
!pip install -U google-genai

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

In [ ]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
import time
import google.generativeai as genai # Using google.generativeai explicitly

def count_tokens(text: str, model_name: str) -> int:
    # Get a generative model instance
    model_instance = genai.GenerativeModel(model_name)
    # Call the count_tokens method on the model instance
    response = model_instance.count_tokens(text)
    return response.total_tokens

def safe_count_tokens(
    text: str,
    model: str = "gemini-2.5-flash",
    max_retries: int = 5,
    wait_seconds: int = 60
) -> int | None:

    for attempt in range(max_retries):
        try:
            return count_tokens(text, model_name=model)

        except Exception as e:
            msg = str(e).lower()

            # detect quota / rate limit issues
            if "resource exhausted" in msg or "quota" in msg or "429" in msg:
                print(f"[WARN] Quota hit. Waiting {wait_seconds}s before retry... (attempt {attempt+1})")
                time.sleep(wait_seconds)
            else:
                print(f"[ERROR] Token count failed: {e}")
                return None

    print("[ERROR] Max retries exceeded for token counting")
    return None

In [ ]:
# Test safe_count_tokens with a single example from the dataset
# This will help verify if the token counting mechanism is working correctly.

# Ensure dataset_with_embedding_text is available, assuming previous cells ran correctly.
if 'dataset_with_embedding_text' in locals() or 'dataset_with_embedding_text' in globals():
    single_example_text = dataset_with_embedding_text['train'][0]['embedding_text']
    single_example_token_count = safe_count_tokens(single_example_text)

    print(f"Embedding Text for first example: {single_example_text[:200]}...") # Print a snippet
    print(f"Token count for first example: {single_example_token_count}")
else:
    print("Error: 'dataset_with_embedding_text' is not defined. Please run all preceding cells.")

Embedding Text for first example: title: The Te Aroha hot springs (mainly in the nineteenth century)

authors: Hart, Philip

abstract: Unique amongst New Zealand mining districts, the hot springs sited at the base of the mountain were...
Token count for first example: 562


In [56]:
def add_token_count_to_dataset(example):
    # Skip if token count already exists and is not None
    if (
        'embedding_token_count' in example
        and example['embedding_token_count'] is not None
    ):
        return example

    example['embedding_token_count'] = safe_count_tokens(
        example['embedding_text']
    )

    return example

# Apply the safe_count_tokens function to the 'embedding_text' column using dataset.map
dataset_with_embedding_text = dataset_with_embedding_text.map(add_token_count_to_dataset)

print("Train Dataset with token counts (first 5 rows):")
display(dataset_with_embedding_text['train'].select(range(5)).select_columns(['embedding_text', 'embedding_token_count']))

print("\nTest Dataset with token counts (first 5 rows):")
display(dataset_with_embedding_text['test'].select(range(5)).select_columns(['embedding_text', 'embedding_token_count']))

Map:   0%|          | 0/13271 [00:00<?, ? examples/s]

Map:   0%|          | 0/3318 [00:00<?, ? examples/s]

Train Dataset with token counts (first 5 rows):


Dataset({
    features: ['embedding_text', 'embedding_token_count'],
    num_rows: 5
})


Test Dataset with token counts (first 5 rows):


Dataset({
    features: ['embedding_text', 'embedding_token_count'],
    num_rows: 5
})

In [57]:
print(dataset['train'])
print(dataset['test'])

print(dataset['train'].column_names)
print(dataset['test'].column_names)

print(dataset['train'][0])

Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence'],
    num_rows: 13271
})
Dataset({
    features: ['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence'],
    num_rows: 3318
})
['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence']
['title', 'authors', 'subjects', 'abstract', 'text', 'record_id_hash', 'prompt', 'token_count', 'classification_label', 'classification_reason', 'year', 'classification_confidence']
{'title': 'The Te Aroha hot springs (mainly in the nineteenth century)', 'authors': 'Hart, Philip', 'subjects': None, 'abstract': 'Unique amongst New Zealand mining districts, t

In [58]:
print("Train rows:", len(dataset_with_embedding_text['train']))
print("Test rows:", len(dataset_with_embedding_text['test']))

print("Total rows:", len(dataset_with_embedding_text['train']) + len(dataset_with_embedding_text['test']))

Train rows: 13271
Test rows: 3318
Total rows: 16589


In [59]:
failed_rows = sum(
    1 for x in dataset_with_embedding_text['train']["embedding_token_count"]
    if x is None or x == 0
)

successful_rows = sum(
    1 for x in dataset_with_embedding_text['train']["embedding_token_count"]
    if x is not None and x > 0
)

print("Failed rows (train split):", failed_rows)
print("Successful rows (train split):", successful_rows)
print("Total rows (train split):", len(dataset_with_embedding_text['train']))

Failed rows (train split): 0
Successful rows (train split): 13271
Total rows (train split): 13271


In [60]:
failed_rows_test = sum(
    1 for x in dataset_with_embedding_text['test']["embedding_token_count"]
    if x is None or x == 0
)

successful_rows_test = sum(
    1 for x in dataset_with_embedding_text['test']["embedding_token_count"]
    if x is not None and x > 0
)

print("Failed rows (test split):", failed_rows_test)
print("Successful rows (test split):", successful_rows_test)
print("Total rows (test split):", len(dataset_with_embedding_text['test']))

Failed rows (test split): 0
Successful rows (test split): 3318
Total rows (test split): 3318


### Push to Hugging Face Hub

To push your dataset to the Hugging Face Hub, you need to authenticate. Follow these steps:

1.  Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
2.  Generate a new token with **"write"** role.
3.  Run the cell below and paste your token when prompted.

In [63]:
from huggingface_hub import notebook_login

notebook_login()

In [62]:
hf_dataset_repo_id = "dinushiTJ/nz_research_commons_classification"

dataset_with_embedding_text.push_to_hub(hf_dataset_repo_id)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/14 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|#########9| 63.3MB / 63.6MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 16.1MB / 16.1MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/dinushiTJ/nz_research_commons_classification/commit/33a3627429ec54e7522dd5fcb403eaea96c23867', commit_message='Upload dataset', commit_description='', oid='33a3627429ec54e7522dd5fcb403eaea96c23867', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/dinushiTJ/nz_research_commons_classification', endpoint='https://huggingface.co', repo_type='dataset', repo_id='dinushiTJ/nz_research_commons_classification'), pr_revision=None, pr_num=None)